# 第13回 ソルバ3：式からコードへ

第12回の定式化を実行し、結果を検算します。必修は約70分、任意の節点カバーは約20分です。小レポートはありません。

Colabの「ファイル → ドライブにコピーを保存」で自分用に保存し、上から順に実行してください。予想を書いてから解答セルを開いてください。

In [ ]:
%pip -q install pulp==3.3.0 highspy

In [ ]:
import pulp
import matplotlib.pyplot as plt
from itertools import product

def solve(model):
    model.solve(pulp.HiGHS(msg=False))
    print("状態:", pulp.LpStatus[model.status])
    if model.status != pulp.LpStatusOptimal:
        raise RuntimeError("最適解が得られていません。条件を確認してください。")

## 1 最短経路（約20分）
実在する辺だけに変数を用意します。辺の数字は長さです。始点1、終点5。流出−流入は始点で1、終点で−1、他で0です。

In [ ]:
edges = [[1, 2, 30], [1, 4, 20], [1, 5, 120], [2, 3, 70], [4, 3, 50], [3, 5, 30], [4, 5, 90]]
V = sorted({v for a,b,w in edges for v in (a,b)})
E = [(a,b) for a,b,w in edges]
length = {(a,b):w for a,b,w in edges}
model = pulp.LpProblem("shortest_path", pulp.LpMinimize)
x = pulp.LpVariable.dicts("x", E, cat="Binary")
model += pulp.lpSum(length[e]*x[e] for e in E)
for v in V:
    supply = 1 if v == 1 else (-1 if v == 5 else 0)
    model += (pulp.lpSum(x[a,b] for a,b in E if a == v)
              - pulp.lpSum(x[a,b] for a,b in E if b == v) == supply)
solve(model)
print("最短距離:", pulp.value(model.objective))
print("使用辺:", [e for e in E if x[e].value() > 0.5])
# 今回の正の長さ・0–1モデルでは、選択辺から1本の最短経路を読める。
selected_edges = [e for e in E if x[e].value() > 0.5]
path = [1]
while path[-1] != 5:
    next_vertices = [b for a, b in selected_edges if a == path[-1]]
    assert len(next_vertices) == 1, "選択辺と頂点の収支を確認してください。"
    nxt = next_vertices[0]
    assert nxt not in path, "閉路が含まれています。モデルと辺の長さを確認してください。"
    path.append(nxt)
path_length = sum(length[a, b] for a, b in zip(path, path[1:]))
print("経路:", " → ".join(map(str, path)))
print("経路長の検算:", path_length)
assert abs(path_length - pulp.value(model.objective)) < 1e-7

### 練習1
各頂点で「流出−流入」を計算して、保存式を満たすか確認してください。解が複数ある場合、辺の組が一つの解答と違うだけでは誤りではありません。

In [ ]:
# 自分で検算を書く


<details><summary>解答と検算</summary>このコードは cat="Binary" を指定しているので、各辺の変数は0か1です。選択辺を始点からたどると1→4→3→5となり、長さは20＋50＋30＝100です。各頂点の収支と、各辺の0–1条件も確認します。

補足：0–1条件を外して非負の連続変数にしたLPでも、この最短経路問題では整数の最適解が存在します。ただし、そのLPの最適解がすべて整数とは限りません。これは、上の0–1モデルとは区別する説明です。</details>

In [ ]:
for v in V:
    balance = sum(x[a,b].value() for a,b in E if a == v)-sum(x[a,b].value() for a,b in E if b == v)
    expected = 1 if v == 1 else (-1 if v == 5 else 0)
    assert abs(balance-expected)<1e-7
    print(v, balance)
assert all(x[e].value() >= -1e-7 for e in E)

### 追加練習：経路が存在しない場合
終点5へ入る辺をすべて除くとどうなるか、予想してから状態を確認します。

In [ ]:
remaining=[(a,b) for a,b in E if b!=5]
broken=pulp.LpProblem("no_path",pulp.LpMinimize)
z=pulp.LpVariable.dicts("z",remaining,cat="Binary")
broken += pulp.lpSum(length[e]*z[e] for e in remaining)
for v in V:
    rhs=1 if v==1 else -1 if v==5 else 0
    broken += pulp.lpSum(z[a,b] for a,b in remaining if a==v)-pulp.lpSum(z[a,b] for a,b in remaining if b==v)==rhs
broken.solve(pulp.HiGHS(msg=False))
print(pulp.LpStatus[broken.status])
assert broken.status==pulp.LpStatusInfeasible

解説：終点に必要な流入1を供給できません。Infeasibleは条件を満たす解がないという状態で、得られた数値を最適解として読んではいけません。

## 2 最大流（約20分）
第10・11回のネットワークを使います。変数は辺の流量と全体の流量Fです。長さの最小化と容量制約の違いを確認します。

In [ ]:
flow_edges = [[1, 2, 7], [1, 3, 8], [2, 3, 4], [2, 4, 5], [3, 4, 3], [3, 5, 3], [4, 5, 4], [4, 6, 6], [5, 6, 9]]
FE = [(a,b) for a,b,cap in flow_edges]
capacity = {(a,b):cap for a,b,cap in flow_edges}
flow_model = pulp.LpProblem("max_flow", pulp.LpMaximize)
f = {e:pulp.LpVariable(f"f_{e[0]}_{e[1]}", 0, capacity[e]) for e in FE}
F = pulp.LpVariable("F", lowBound=0)
flow_model += F
for v in range(1,7):
    net = pulp.lpSum(f[a,b] for a,b in FE if a == v)-pulp.lpSum(f[a,b] for a,b in FE if b == v)
    flow_model += net == (F if v == 1 else (-F if v == 6 else 0))
solve(flow_model)
print("最大流量:", F.value())
for e in FE: print(e, f[e].value(), "/", capacity[e])

### 練習2
最大流量が11になることを確かめ、容量・保存式を検算してください。辺ごとの流量が手計算と違ってもよい理由を考えてください。

In [ ]:
# 自分で検算を書く


<details><summary>解答</summary>同じ目的値を与える複数の流量があり得ます。容量と保存式を満たし、カットS={1,2,3}の容量11と流量が一致すれば最大です。</details>

In [ ]:
assert all(-1e-7 <= f[e].value() <= capacity[e]+1e-7 for e in FE)
for v in range(1,7):
    net = sum(f[a,b].value() for a,b in FE if a==v)-sum(f[a,b].value() for a,b in FE if b==v)
    assert abs(net-(F.value() if v==1 else -F.value() if v==6 else 0))<1e-7
S={1,2,3}
cut=sum(capacity[a,b] for a,b in FE if a in S and b not in S)
assert abs(F.value()-cut)<1e-7
print("カット上限と一致:", cut)

## 3 ナップサック（約30分）
第12回と同じ品物です。予算100円、各品物1個まで、分割できません。0–1変数で選択を表します。

In [ ]:
prices = [60, 30, 40, 50, 40, 30]
values = [36, 27, 12, 50, 28, 24]
budget = 100

def solve_knapsack(budget):
    model = pulp.LpProblem("snacks", pulp.LpMaximize)
    x = pulp.LpVariable.dicts("take", range(len(prices)), cat="Binary")
    model += pulp.lpSum(values[i]*x[i] for i in x)
    model += pulp.lpSum(prices[i]*x[i] for i in x) <= budget
    solve(model)
    chosen = [i for i in x if x[i].value()>0.5]
    cost = sum(prices[i] for i in chosen)
    value = sum(values[i] for i in chosen)
    assert cost <= budget
    print("品物:", [i+1 for i in chosen], "価格:", cost, "満足度:", value)
    return chosen, cost, value

chosen, cost, value = solve_knapsack(budget)
assert value == 79

In [ ]:
plt.bar(range(1,7), values, color=["seagreen" if i in chosen else "lightgray" for i in range(6)])
plt.xlabel("item"); plt.ylabel("value"); plt.show()

### 練習3
予算を90円に変えると何を選ぶでしょうか。100円の答えから一品を捨てれば最適になるか、予想してから実行してください。

In [ ]:
# budget を変更して実行する


<details><summary>解答</summary>品物4と5、価格90円、満足度78です。最適な選択は予算によって組み替わります。満足度の比だけで決める方法では0–1問題の最適性を保証できません。</details>

In [ ]:
chosen90, cost90, value90 = solve_knapsack(90)
assert value90 == 78
# 小さい例なので全64通りで独立に検算できる
for B in (90,100):
    candidates = [(sum(u*z for u,z in zip(values,bits)),bits) for bits in product((0,1),repeat=6) if sum(p*z for p,z in zip(prices,bits))<=B]
    print(B, max(v for v,bits in candidates))

## 任意：節点カバー（約20分）
各辺の端点を少なくとも一つ選び、選ぶ頂点数を最小化します。第12回のモデルをコードにします。

In [ ]:
vertices=[1,2,3,4]
cover_edges=[(1,2),(2,3),(3,4),(1,4),(2,4)]
cm=pulp.LpProblem("vertex_cover",pulp.LpMinimize)
y=pulp.LpVariable.dicts("choose",vertices,cat="Binary")
cm += pulp.lpSum(y.values())
for a,b in cover_edges: cm += y[a]+y[b]>=1
solve(cm)
selected={v for v in vertices if y[v].value()>0.5}
assert all(a in selected or b in selected for a,b in cover_edges)
print("選択:",selected,"個数:",len(selected))

## 次回へ
ソルバで求めたナップサックの最適値79を、次回は動的計画法の表で求めます。変数・目的・制約・検算を自分の言葉で説明してください。